## **Task 3. Extrapolation**

**Points: 2**

**Forecast the 200 missing data points at the end of each series. Indicate the uncertainty of your estimate for each point!**
• Evaluate how well your proposed solutions are expected to work.
• Make sure to avoid overfitting and data leakage in your models.
• Can you use insights from Task 1 to improve extrapolation?
<br>
You are free to choose which methods to use for task 2 and 3. There are no rules other than the
obvious one that the methods must be statistically sound. It is probably best to use one method
for the interpolation problem (when data are missing within a series) and another method for the
extrapolation problem (when data is missing at the end of the series).


## Method: PCA + VAR(2) Extrapolation

**Why this approach?**

- **PCA** reduces the 7 correlated price series to a small set of uncorrelated factor series.  
  Eigenvalues / explained-variance ratios motivate how many factors to keep.  
  *In this dataset PC1 alone explains ≈ 99.6 % of the variance in log-returns — the 7 series  
  are nearly perfectly co-integrated by a single common factor (a "market-wide" level).  
  We enforce a minimum of 2 components so the model remains multivariate (VAR requires ≥ 2 series);  
  the additional component captures residual cross-series structure.*
- **VAR(2)** (Vector Autoregression, lag 2) models the joint short-term dynamics of the  
  PCA factors. Lag order 2 is fixed deliberately for parsimony and to avoid overfitting.

**Pipeline (strict no-leakage)**

| Step | What | Data used |
|------|------|-----------|
| 1 | Extract observed prices → log-prices → log-returns | observed window only |
| 2 | Linear-interpolate 50 internal NaN gaps | within training window only |
| 3 | Standardise returns, fit PCA | training data only |
| 4 | Project returns to PCA scores, fit VAR(2) | training data only |
| 5 | Forecast 200 steps in PCA space, back-transform | no hold-out info used |
| 6 | Bootstrap 1 000 paths (resample VAR residuals) | no hold-out info used |

**Hold-out validation** removes the last 200 *observed* return-steps as a proxy unseen test set  
to evaluate forecast quality before producing the real forecast.


In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from statsmodels.tsa.api import VAR
from sklearn.metrics import mean_squared_error, mean_absolute_error

warnings.filterwarnings('ignore')
np.random.seed(42)
%matplotlib inline
plt.rcParams.update({'figure.dpi': 100, 'axes.grid': True, 'grid.alpha': 0.3,
                     'figure.figsize': (14, 4)})


In [ ]:
# Relative path: notebooks/ -> data/
DATA_PATH = os.path.join(os.path.dirname(os.path.abspath('__file__')), '..', 'data', 'spiff_data-2.csv')
if not os.path.exists(DATA_PATH):
    DATA_PATH = '../data/spiff_data-2.csv'

df_raw     = pd.read_csv(DATA_PATH, index_col=0)
PRICE_COLS = ['gurkor', 'guitars', 'slingshots', 'stocks', 'sugar', 'water', 'tranquillity']

print(f"Raw data shape : {df_raw.shape}")
print(f"NaN per series:\n{df_raw[PRICE_COLS].isna().sum().to_frame('NaN count').T}")

# Last observed row index (before the trailing NaN block)
last_obs_idx = int(df_raw[PRICE_COLS[0]].dropna().index[-1])
n_forecast   = len(df_raw) - last_obs_idx - 1
print(f"\nLast observed row index : {last_obs_idx}  (day {df_raw.loc[last_obs_idx, 'day']})")
print(f"Forecast horizon        : {n_forecast} steps")


## Data Preparation

1. **Observed window** – rows 0 → `last_obs_idx` (5 206 price observations per series).
2. **Internal NaN gaps** (50 per series) – filled with linear interpolation *inside the  
   observed window only*. No information from the forecast period is used.
3. **Log-prices** $\ell_t = \ln p_t$ – additive and more variance-stable.
4. **Log-returns** $r_t = \ell_t - \ell_{t-1}$ – approximately stationary; used as VAR inputs.


In [ ]:
# ── Observed price matrix (before the 200-step gap) ──────────────────────────
prices_obs = df_raw[PRICE_COLS].iloc[:last_obs_idx + 1].copy()

# ── Fill internal NaN gaps (linear interpolation, no leakage) ────────────────
prices_obs_filled = prices_obs.interpolate(method='linear', limit_direction='both')
assert prices_obs_filled.isna().sum().sum() == 0, "Unexpected NaNs remain after interpolation"

# ── Log-prices and log-returns ────────────────────────────────────────────────
log_prices_obs = np.log(prices_obs_filled)                # shape (last_obs_idx+1, 7)
log_returns_obs = log_prices_obs.diff().dropna()          # shape (last_obs_idx, 7)

print(f"Observed prices  : {prices_obs.shape}")
print(f"Log-prices       : {log_prices_obs.shape}")
print(f"Log-returns      : {log_returns_obs.shape}")
print(f"\nLog-return descriptives:\n{log_returns_obs.describe().round(5)}")


## Hold-out Validation

The last 200 *observed* return-steps are removed and treated as an unseen test set.

- **Training window** – all log-returns except the final 200.  
  PCA is fitted and standardisation is applied **only** to this window.
- **Test window** – the final 200 observed return-steps, used solely for evaluation.

Metrics: RMSE, MAE, MAPE per series.


In [ ]:
HOLDOUT    = 200
N_RETURNS  = len(log_returns_obs)

# ── Train / hold-out split ────────────────────────────────────────────────────
train_ret = log_returns_obs.iloc[: N_RETURNS - HOLDOUT].copy()   # (N-200, 7)
hold_ret  = log_returns_obs.iloc[N_RETURNS - HOLDOUT :].copy()   # (200,   7)

# Last log-price in the training window (back-transform anchor)
last_lp_ho = log_prices_obs.iloc[N_RETURNS - HOLDOUT]            # (7,)

print(f"Training returns : {train_ret.shape}")
print(f"Hold-out returns : {hold_ret.shape}")

# ── PCA fit (training data ONLY) ──────────────────────────────────────────────
scaler_ho  = StandardScaler()
train_std  = scaler_ho.fit_transform(train_ret)

pca_ho     = PCA()
pca_ho.fit(train_std)

evr    = pca_ho.explained_variance_ratio_
cumevr = np.cumsum(evr)
# Minimum 2 components so VAR stays multivariate; 95% EV as default floor
K_HO   = max(2, int(np.searchsorted(cumevr, 0.95)) + 1)

print(f"\nEigenvalues            : {np.round(pca_ho.explained_variance_, 4)}")
print(f"Explained variance     : {np.round(evr, 4)}")
print(f"Cumulative EV          : {np.round(cumevr, 4)}")
print(f"Components kept        : {K_HO}  (min 2, ≥95% EV rule)")

# ── Scree plot ────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(range(1, len(evr)+1), evr * 100, color='steelblue')
axes[0].set_xlabel('Component'); axes[0].set_ylabel('Explained variance (%)'); axes[0].set_title('Scree plot')
axes[1].plot(range(1, len(cumevr)+1), cumevr * 100, marker='o', color='steelblue')
axes[1].axhline(95, color='red', linestyle='--', lw=1, label='95 %')
axes[1].set_xlabel('Components'); axes[1].set_ylabel('Cumulative EV (%)'); axes[1].set_title('Cumulative explained variance')
axes[1].legend()
plt.tight_layout(); plt.show()

# PCA factor scores (training)
scores_train_ho = pca_ho.transform(train_std)[:, :K_HO]
scores_df_ho    = pd.DataFrame(scores_train_ho, columns=[f'PC{i+1}' for i in range(K_HO)])

# ── VAR(2) fit on PCA factors (training data ONLY) ───────────────────────────
var_model_ho = VAR(scores_df_ho)
var_fit_ho   = var_model_ho.fit(2)    # explicit lag-2

print(f"\nVAR(2) hold-out model  AIC={var_fit_ho.aic:.4f}  BIC={var_fit_ho.bic:.4f}  lag={var_fit_ho.k_ar}")


In [ ]:
def back_transform_paths(paths, pca_model, scaler, n_components, last_log_price_vec):
    """
    Inverse-transform PCA paths back to price level.

    paths              : ndarray (n_paths, steps, n_components)
    pca_model          : fitted sklearn PCA
    scaler             : fitted StandardScaler
    n_components       : int – how many PCA components were kept
    last_log_price_vec : ndarray (7,) – last observed log-price per series
    Returns
    -------
    prices : ndarray (n_paths, steps, 7)
    """
    n_paths, steps, _ = paths.shape
    n_total = pca_model.n_components_

    # Pad discarded components with zeros before inverse-PCA
    full = np.zeros((n_paths, steps, n_total))
    full[:, :, :n_components] = paths

    # Inverse PCA  →  standardised log-returns
    std_ret = full @ pca_model.components_               # (n_paths, steps, 7)

    # Inverse standardisation  →  log-returns
    log_ret = std_ret * scaler.scale_ + scaler.mean_     # broadcast over paths/steps

    # Cumulative sum  →  log-prices, anchored at last_log_price_vec
    log_p = np.cumsum(log_ret, axis=1) + last_log_price_vec   # (n_paths, steps, 7)

    return np.exp(log_p)

# ── Hold-out point forecast ───────────────────────────────────────────────────
fc_input_ho  = scores_df_ho.values[-var_fit_ho.k_ar:]    # (2, K_HO)
fc_scores_ho = var_fit_ho.forecast(fc_input_ho, steps=HOLDOUT)   # (200, K_HO)

fc_prices_ho = back_transform_paths(
    fc_scores_ho[np.newaxis], pca_ho, scaler_ho, K_HO, last_lp_ho.values
)[0]   # (200, 7)

# True hold-out prices (back-transform the true returns)
true_lp_ho     = np.cumsum(hold_ret.values, axis=0) + last_lp_ho.values
true_prices_ho = np.exp(true_lp_ho)   # (200, 7)

# ── Evaluation metrics ────────────────────────────────────────────────────────
rows = []
for j, col in enumerate(PRICE_COLS):
    pred = fc_prices_ho[:, j]
    true = true_prices_ho[:, j]
    rows.append({
        'series'  : col,
        'RMSE'    : np.sqrt(mean_squared_error(true, pred)),
        'MAE'     : mean_absolute_error(true, pred),
        'MAPE (%)': np.mean(np.abs((true - pred) / true)) * 100,
    })

metrics_df = pd.DataFrame(rows).set_index('series')
print("Hold-out evaluation (last 200 observed steps):\n")
print(metrics_df.round(4))


In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(16, 20))
axes = axes.ravel()

for j, col in enumerate(PRICE_COLS):
    ax = axes[j]
    ax.plot(true_prices_ho[:, j], label='True',     color='black',     lw=1.5)
    ax.plot(fc_prices_ho[:,  j], label='Forecast',  color='steelblue', lw=1.5, linestyle='--')
    ax.set_title(f'{col}   RMSE={metrics_df.loc[col,"RMSE"]:.3f}   MAPE={metrics_df.loc[col,"MAPE (%)"]:.2f}%')
    ax.set_xlabel('Hold-out step')
    ax.set_ylabel('Price')
    ax.legend()

axes[-1].axis('off')
fig.suptitle('Hold-out Validation: True vs PCA+VAR(2) Forecast (200 steps)', fontsize=14)
plt.tight_layout()
plt.show()


## Final Forecast: 200 Missing Steps

With hold-out performance established, we retrain on **all 5 206 observed data points**  
(full observed window) to produce the final forecast.

**Uncertainty quantification via residual bootstrap:**  
1. Fit VAR(2) on all training PCA scores.  
2. Extract VAR residuals.  
3. Simulate 1 000 paths: at each step, apply VAR coefficients and add a randomly sampled  
   (with replacement) residual from the fitted model's residual matrix.  
4. Back-transform each path to price level.  
5. Report 10th/90th percentiles → 80 % PI; 2.5th/97.5th → 95 % PI.


In [ ]:
# ── Fit PCA on ALL observed log-returns ──────────────────────────────────────
scaler_final = StandardScaler()
all_std      = scaler_final.fit_transform(log_returns_obs)

pca_final    = PCA()
pca_final.fit(all_std)

evr_f    = pca_final.explained_variance_ratio_
cumevr_f = np.cumsum(evr_f)
# Minimum 2 components so VAR stays multivariate; 95% EV as default floor
K_FINAL  = max(2, int(np.searchsorted(cumevr_f, 0.95)) + 1)

print(f"Final model – components kept (min 2, >=95% EV): {K_FINAL}")
print(f"Eigenvalues         : {np.round(pca_final.explained_variance_, 4)}")
print(f"Explained variance  : {np.round(evr_f, 4)}")
print(f"Cumulative EV       : {np.round(cumevr_f, 4)}")

scores_final    = pca_final.transform(all_std)[:, :K_FINAL]
scores_final_df = pd.DataFrame(scores_final, columns=[f'PC{i+1}' for i in range(K_FINAL)])

# ── Fit VAR(2) on all PCA factors ────────────────────────────────────────────
var_model_final = VAR(scores_final_df)
var_fit_final   = var_model_final.fit(2)   # explicit VAR(2)

print(f"\nFinal VAR(2)   AIC={var_fit_final.aic:.4f}   BIC={var_fit_final.bic:.4f}   lag={var_fit_final.k_ar}")

# Last log-price before the forecast gap (anchor for back-transform)
last_lp_final = log_prices_obs.iloc[-1]
print(f"\nLast observed log-price:\n{last_lp_final.round(4)}")


In [ ]:
N_SIM = 1000
H     = 200
LAG   = var_fit_final.k_ar     # = 2

coefs     = var_fit_final.coefs                        # (2, K_FINAL, K_FINAL)
intercept = var_fit_final.intercept                    # (K_FINAL,)
resids    = np.asarray(var_fit_final.resid)            # (T-2, K_FINAL) as numpy array

# Initialisation: last LAG PCA score vectors
init_scores = scores_final_df.values[-LAG:]   # (2, K_FINAL)

# ── Bootstrap simulation ─────────────────────────────────────────────────────
sim_paths = np.zeros((N_SIM, H, K_FINAL))

for s in range(N_SIM):
    idx     = np.random.choice(len(resids), size=H, replace=True)
    eps     = resids[idx]                   # (H, K_FINAL)
    history = list(init_scores)             # mutable; grows by 1 each step
    for t in range(H):
        z_new = intercept.copy()
        for lag_i in range(LAG):
            z_new += coefs[lag_i] @ history[-1 - lag_i]
        z_new += eps[t]
        history.append(z_new)
    sim_paths[s] = np.array(history[LAG:])   # (H, K_FINAL)

# ── Deterministic point forecast ─────────────────────────────────────────────
fc_det = var_fit_final.forecast(init_scores, steps=H)   # (H, K_FINAL)

fc_prices_final = back_transform_paths(
    fc_det[np.newaxis], pca_final, scaler_final, K_FINAL, last_lp_final.values
)[0]   # (H, 7)

# ── Back-transform all bootstrap paths ───────────────────────────────────────
sim_prices = back_transform_paths(
    sim_paths, pca_final, scaler_final, K_FINAL, last_lp_final.values
)  # (N_SIM, H, 7)

# ── Prediction intervals ─────────────────────────────────────────────────────
PI = {
    'lower_95': np.percentile(sim_prices,  2.5, axis=0),   # (H, 7)
    'upper_95': np.percentile(sim_prices, 97.5, axis=0),
    'lower_80': np.percentile(sim_prices, 10.0, axis=0),
    'upper_80': np.percentile(sim_prices, 90.0, axis=0),
}

print(f"Bootstrap complete: {N_SIM} paths × {H} steps × 7 series")
print("\nPoint forecast (first 5 steps):")
print(pd.DataFrame(fc_prices_final[:5], columns=PRICE_COLS).round(4))


In [ ]:
CONTEXT = 100   # show last 100 observed observations for context

fig, axes = plt.subplots(4, 2, figsize=(16, 22))
axes = axes.ravel()
steps = np.arange(1, H + 1)

for j, col in enumerate(PRICE_COLS):
    ax = axes[j]

    # Historical context
    hist_x = np.arange(-CONTEXT, 0)
    ax.plot(hist_x, prices_obs_filled[col].values[-CONTEXT:],
            color='black', lw=1.2, label='Observed')

    # Prediction intervals
    ax.fill_between(steps,
                    PI['lower_95'][:, j], PI['upper_95'][:, j],
                    alpha=0.15, color='steelblue', label='95% PI')
    ax.fill_between(steps,
                    PI['lower_80'][:, j], PI['upper_80'][:, j],
                    alpha=0.32, color='steelblue', label='80% PI')

    # Point forecast
    ax.plot(steps, fc_prices_final[:, j],
            color='steelblue', lw=1.5, linestyle='--', label='Point forecast')

    ax.axvline(x=0.5, color='gray', linestyle=':', lw=1)
    ax.set_title(col, fontsize=11)
    ax.set_xlabel('Step  (0 = last observed,  1..200 = forecast)')
    ax.set_ylabel('Price')
    ax.legend(fontsize=8)

axes[-1].axis('off')
fig.suptitle('Final Forecast: PCA + VAR(2) with Bootstrap Prediction Intervals', fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
# ── Build forecast_task3 DataFrame ───────────────────────────────────────────
records = []
for step_i in range(H):
    for j, col in enumerate(PRICE_COLS):
        records.append({
            'step'          : step_i + 1,
            'series'        : col,
            'forecast_price': fc_prices_final[step_i, j],
            'lower_80'      : PI['lower_80'][step_i, j],
            'upper_80'      : PI['upper_80'][step_i, j],
            'lower_95'      : PI['lower_95'][step_i, j],
            'upper_95'      : PI['upper_95'][step_i, j],
        })

forecast_task3 = pd.DataFrame(records)
print(f"forecast_task3  shape: {forecast_task3.shape}")
print(f"\nFirst 14 rows:\n{forecast_task3.head(14).to_string(index=False)}")
print(f"\nLast 7 rows:\n{forecast_task3.tail(7).to_string(index=False)}")
